# Let's build a data agent

We will use data from the 2026 worldcup. The data contains 11 related tables: matches, teams, players, goals, bookings, lineups, substitutions, referees, match referees, penalty shootouts, and tournaments.

## Part 1 — Set up the workshop

`smolagents` supplies the agent loop, LiteLLM connects it to OpenAI, and DuckDB is the warehouse.

In [ ]:
%pip install -q "smolagents==1.26.0" litellm duckdb pandas tabulate ipywidgets

In [ ]:
import getpass
import os
import re
from pathlib import Path
from notebook_ui import run_agent
import duckdb
from smolagents import LiteLLMModel, tool
from agent import ToolCallingAgent

### Connect to the supplied database

Keep the `.duckdb` file beside this notebook. The fallback path also works in Google Colab after uploading the file to `/content`.

In [ ]:
candidates = [Path("worldcup-2026.duckdb"), Path("/content/worldcup-2026.duckdb")]
DB_PATH = next((p for p in candidates if p.exists()), None)
assert DB_PATH is not None, "Place worldcup-2026.duckdb beside the notebook (or upload it to /content in Colab)."

# Read-only at the connection level, not merely in the prompt.
con = duckdb.connect(str(DB_PATH), read_only=True)
TABLES = [row[0] for row in con.execute("SHOW TABLES").fetchall()]

print("database:", DB_PATH.resolve())
print("tables:", TABLES)
print("matches:", con.execute("SELECT count(*) FROM matches").fetchone()[0])

### Configure the model

If `OPENAI_API_KEY` already exists in the notebook environment, this uses it. Otherwise you will be prompted securely; the key is not printed or stored in the notebook.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: " )

MODEL_ID = "openai/gpt-5-mini"
model = LiteLLMModel(
    model_id=MODEL_ID,
    api_key=os.environ["OPENAI_API_KEY"],
    max_tokens=1200,
    reasoning_effort="low",
    reasoning_summary="auto",
)
print("model:", MODEL_ID)

## Part 2 — Create the Agent with 2 core tools

In [ ]:
MAX_ROWS = 50

@tool
def list_tables() -> str:
    """List the tables available in the World Cup warehouse."""
    return "\n".join(TABLES)

@tool
def execute_sql(query: str) -> str:
    """Call this tool to execute SQL.

    Args:
        query: One SELECT query.
    """
    try:
        df = con.execute(query).df()
    except Exception:
        return "Query failed."
#    except Exception as exc:
#        return f"ERROR: {type(exc).__name__}: {exc}"
    if df.empty:
        return "Query returned 0 rows."
    return df.head(MAX_ROWS).to_markdown(index=False)

In [ ]:
MINIMAL_PROMPT = """You are a data analyst answering from a DuckDB warehouse.
Use your tools to answer questions. Keep the final answer concise."""
def make_agent(tools, instructions, max_steps=10, agent_model=None):
    return ToolCallingAgent(
        tools=tools,
        model=agent_model or model,
        instructions=instructions,
        max_steps=max_steps,
    )
minimal_agent = make_agent([list_tables, execute_sql], MINIMAL_PROMPT)

### Try out some questions 
- How many matches are in the dataset?
- Who were the five leading goalscorers?
- Which player received the latest card in the tournament?
- Which three players scored the most non-penalty goals, and which teams did
  they represent?
- Who played in the final, who won, what was the final score, and was it decided
  in regulation, extra time, or a penalty shootout?

### Watch the trace, not only the final answer:
- Replay the same questions multiple times. Are they consistent?
- How many steps did it use?
- Did a failed query give it enough information to recover?
- Could a valid SQL give a wrong answer?
- Did it know the schema, or guess column names?
"""

In [ ]:
run_agent(minimal_agent, "Who played in the final, who won, what was the final score, and was it decided in regulation, extra time, or a penalty shootout?")


## Part 3 — Add a compact data profiler

The agent goes straight into querying, gets errors, realizes it needs to explore the schema. Let's help it by adding a specialized tool to get the schema, nulls, and a few actual values. It avoids dumping entire tables into the context window.
The agent doesn't have to discover it all on it's own through failures.

In [ ]:
def _quoted_table(name: str) -> str:
    if name not in TABLES:
        raise ValueError(f"Unknown table {name!r}. Available: {', '.join(TABLES)}")
    return '"' + name.replace('"', '""') + '"'

@tool
def inspect_table(table_name: str) -> str:
    """Profile one table: columns, types, null counts, and example values.

    Args:
        table_name: Exact table name returned by list_tables.
    """
    try:
        table = _quoted_table(table_name)
    except ValueError as exc:
        return f"ERROR: {exc}"
    row_count = con.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
    columns = con.execute(f"DESCRIBE {table}").fetchall()
    lines = [f"table: {table_name}", f"rows: {row_count}", "columns:"]
    for name, dtype, *_ in columns:
        column = '"' + name.replace('"', '""') + '"'
        nulls = con.execute(
            f"SELECT count(*) FILTER (WHERE {column} IS NULL) FROM {table}"
        ).fetchone()[0]
        examples = con.execute(
            f"SELECT DISTINCT {column} FROM {table} WHERE {column} IS NOT NULL LIMIT 4"
        ).fetchall()
        sample = ", ".join(repr(row[0]) for row in examples)
        lines.append(f"- {name}: {dtype}; nulls={nulls}; examples=[{sample}]")
    return "\n".join(lines)

In [ ]:
PROFILE_PROMPT = MINIMAL_PROMPT + """
Before querying, inspect every relevant table. Use observed column names and values.
Resolve internal IDs to human-readable names before answering."""

profile_agent = make_agent(
    [list_tables, inspect_table, execute_sql],
    PROFILE_PROMPT,
)
run_agent(
    profile_agent,
    "Among the three highest-scoring teams, which team received the most ordinary yellow cards? Report its total goals and yellow cards.",
)

### A schema profile still does not explain business meaning

Run this question more than once. The relevant columns are visible, but the agent has not been told whether regulation, extra-time, or shootout scores take precedence.

In [ ]:
FINAL_QUESTION = (
    "Using FIFA fair-play deductions, which team had the worst disciplinary score in this tournament, and what was the score?"
)
run_agent(profile_agent, FINAL_QUESTION)

## Part 4 — Add a data contract

Critical semantics are short enough to place directly in the system instructions. A discoverable documentation tool can still be ignored by the model.

In [ ]:
DATA_CONTRACT = """
World Cup warehouse contract

Relationships
- matches.home_team_id and away_team_id -> teams.team_id
- players.team_id -> teams.team_id
- goals.player_id -> players.player_id
- goals.credited_team_id -> teams.team_id
- lineups, substitutions, and bookings use match_id and team_id

Business rules
- Match user-provided names case-insensitively; source capitalization varies.
- home_team_id and every home_score_* column refer to the same home team;
  away_team_id and every away_score_* column refer to the same away team.
- score_ft is the score after regulation. score_et is the complete score after
  extra time, not goals added during extra time. score_pens is the shootout score.
- To determine a winner, compare score_pens when it is non-null; otherwise compare
  score_et when it is non-null; otherwise compare score_ft.
- goals excludes penalty-shootout attempts.
- For an own goal, player_id is the player who put the ball in their own net;
  credited_team_id is the benefiting team. Exclude own goals from player totals.
- penalty_shootouts has one row per attempt, not one row per match.
- In this import, lineups contains starters plus substitutes who came on.
- card_type values include Y, R, and Y/R.

Join safety
- goals, bookings, lineups, substitutions, and penalty_shootouts are fact tables.
  Aggregate each fact to the required grain before joining facts together.
"""

CONTEXT_PROMPT = PROFILE_PROMPT + "\n" + DATA_CONTRACT
context_agent = make_agent(
    [list_tables, inspect_table, execute_sql],
    CONTEXT_PROMPT,
)

### Run the same semantic question again

In [ ]:
run_agent(context_agent, FINAL_QUESTION)

**Ground truth:** Spain played Argentina. Spain won 1–0 after extra time.

This is independently confirmed by [FIFA’s final tournament standings](https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/articles/final-tournament-standings). The profile exposed the columns; the contract supplied their precedence and meaning.

## Part 5 — Make the expected behaviour executable

The eval below is intentionally small: one question and four required facts. It is enough to catch a model or prompt change that loses the final’s teams, winner, score, or decision method.

In [ ]:
EVAL_MODEL_ID = "openai/gpt-4.1-nano"  # Try "openai/gpt-4.1-nano" after this passes.

eval_model = model if EVAL_MODEL_ID == MODEL_ID else LiteLLMModel(
    model_id=EVAL_MODEL_ID,
    api_key=os.environ["OPENAI_API_KEY"],
    max_tokens=1200,
)
eval_agent = make_agent(
    [list_tables, inspect_table, execute_sql],
    CONTEXT_PROMPT,
    max_steps=5,
    agent_model=eval_model,
)

answer = run_agent(eval_agent, FINAL_QUESTION, return_answer=True)
normalized = str(answer).lower().replace("–", "-")
checks = {
    "mentions both finalists": "spain" in normalized and "argentina" in normalized,
    "names Spain as winner": any(phrase in normalized for phrase in (
        "spain won", "winner was spain", "winner: spain", "winner is spain"
    )),
    "reports the 1-0 score": bool(re.search(r"1\s*-\s*0", normalized)),
    "says extra time": "extra time" in normalized,
}

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", name)
print("\nOVERALL:", "PASS" if all(checks.values()) else "FAIL")

### Change the model

After the eval passes with `MODEL_ID`, change `EVAL_MODEL_ID` to `"openai/gpt-4.1-nano"` and rerun only the eval cell. A cheaper model may fail a required fact or construct the wrong SQL.

One run is not a benchmark. In a real system, run a set of representative cases repeatedly and track pass rate, cost, and latency. The small example establishes the core habit: protect known behaviour before changing the model, prompt, tools, or semantic layer.